# Jailbreak Track
Review grouped out-of-fold detection, Guard labels, refusal behavior, and the frozen manual-audit sample.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from trajectory_extractor import RunStore

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RUN_ID = 'jailbreak-main'
store = RunStore(ROOT / 'runs')
grouped = json.loads((ROOT / 'runs' / RUN_ID / 'metrics' / 'grouped_detection.json').read_text())
grouped['aggregate_oof']


In [ ]:
runs = [store.read(RUN_ID, example_id) for example_id in store.example_ids(RUN_ID)]
pd.DataFrame([{
    'unsafe': run.label,
    'refused': run.provenance.get('refused'),
    'benign': run.provenance.get('benign'),
    'category': run.provenance.get('category'),
} for run in runs]).groupby('benign')[['unsafe', 'refused']].mean()


In [ ]:
audit_dir = ROOT / 'runs' / RUN_ID / 'labels'
sample = json.loads((audit_dir / 'manual_audit_sample.json').read_text())
completed = json.loads((audit_dir / 'manual_audit_completed.json').read_text()) if (audit_dir / 'manual_audit_completed.json').exists() else None
{'sample_size': len(sample), 'completed': completed}


Guard labels are model judgments, not ground truth. Do not report the track as complete until the frozen 20% sample has a recorded human audit.